# Fine-tuning ModernBERT-base on Banking77

**Task**: 77-way intent classification on banking customer queries.

**Recipe**: full fine-tune (not LoRA) with `transformers.Trainer`, bf16 mixed precision, cosine LR schedule with 10% warmup, label-smoothing 0.1, 4 epochs, batch size 32.

**Why this recipe?** ModernBERT-base is 149M params — small enough to fully fine-tune on a free Colab T4. Banking77 has only 13k training examples, so full fine-tuning beats LoRA at this scale. 

In [ ]:
!pip install -q 'transformers>=4.48' 'datasets>=3.1' accelerate scikit-learn evaluate pydantic pydantic-settings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00


In [ ]:
!cd /content/Projects/04-Deep-Learning/text-classification-modernbert

In [ ]:
from datasets import DatasetDict, load_dataset

DATASET_ID = "PolyAI/banking77"
raw = load_dataset(DATASET_ID, revision="refs/pr/6")

split = raw["train"].train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
splits = DatasetDict(train=split["train"], validation=split["test"], test=raw["test"])
names = raw["train"].features["label"].names

print(splits)
print(f'num_labels = {len(names)}')
print(f'example: {splits["train"][0]}')

README.md:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/295k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9002
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1001
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
num_labels = 77
example: {'text': 'I want to use auto top-up. Is there a limit?', 'label': 4}


## The data

Banking77 has a `train`/`test` split out of the box. I carve a 10% stratified-by-label validation slice out of `train` so each of the 77 classes is represented. **The test set is held out and only touched once at the end** — touching it earlier for tuning decisions invalidates the final score.

In [ ]:
# Class balance check — Banking77 isn't perfectly balanced; we want to know.
import pandas as pd
counts = pd.Series(splits['train']['label']).value_counts()
print(f'min class size: {counts.min()}, max: {counts.max()}, median: {counts.median()}')

min class size: 32, max: 168, median: 114.0


## Load the model

`AutoModelForSequenceClassification` wraps the base encoder + a fresh linear head sized for our 77 labels. The base weights are loaded; the head is randomly initialized and trained from scratch.

In [ ]:
from src.model import load_for_training
id2label = dict(enumerate(names))
model, tokenizer = load_for_training(num_labels=len(names), id2label=id2label)
print(f'trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

trainable params: 149,664,077


## Tokenize

In [ ]:
from src.data import tokenize_dataset
train_tok = tokenize_dataset(splits['train'], tokenizer)
val_tok = tokenize_dataset(splits['validation'], tokenizer)
test_tok = tokenize_dataset(splits['test'], tokenizer)
print(train_tok)

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 9002
})


## Train

All knobs live in `src/config.py` so the same recipe is reproducible from the CLI (`python -m src.train`) and the notebook. We select the best checkpoint by macro F1 on the validation set.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
from src.config import settings

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='./outputs/modernbert-banking77',
    num_train_epochs=settings.num_epochs,
    per_device_train_batch_size=settings.per_device_batch_size,
    per_device_eval_batch_size=settings.per_device_batch_size,
    learning_rate=settings.learning_rate,
    weight_decay=settings.weight_decay,
    warmup_ratio=settings.warmup_ratio,
    lr_scheduler_type='cosine',
    label_smoothing_factor=settings.label_smoothing,
    bf16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=25,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.277832,1.205380,0.887113,0.888341
2,1.008923,1.067998,0.924076,0.922019
3,0.857831,1.030009,0.921079,0.918016
4,0.807118,1.026993,0.926074,0.923095


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1128, training_loss=1.2577789569577427, metrics={'train_runtime': 600.1796, 'train_samples_per_second': 59.995, 'train_steps_per_second': 1.879, 'total_flos': 1103365619729076.0, 'train_loss': 1.2577789569577427, 'epoch': 4.0})

In [ ]:
trainer.save_model('./outputs/modernbert-banking77')
tokenizer.save_pretrained('./outputs/modernbert-banking77')
print('Model saved.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


## Final test eval + calibration

Run `python -m src.eval --model_dir ./outputs/modernbert-banking77 --output ./outputs/eval_results.json` from a terminal cell, or run the cells in `03_error_analysis.ipynb` to inspect the confusion matrix and per-class breakdown.

In [ ]:
!python -m scripts.push_to_hub --model_dir ./outputs/modernbert-banking77 --repo_id Ahmed167/modernbert-banking77

Creating repo Ahmed167/modernbert-banking77...
Wrote model card -> outputs/modernbert-banking77/README.md
Uploading model folder...
Found 6 files to upload
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uplo